# 04 — Skew Analysis

Two sections:
1. **Cross-ticker smile comparison** — IV vs moneyness for SPY, AAPL, QQQ, TSLA on one chart
2. **Skew time series** — how ATM IV and put/call skew evolve day-over-day (populates as history accumulates)


In [ ]:
%matplotlib inline
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path


## 1. Cross-ticker IV smile comparison

In [ ]:
# Load today's signals for all four tickers
signal_files = {
    'SPY':  '../data/spy_signals.csv',
    'AAPL': '../data/aapl_signals.csv',
    'QQQ':  '../data/qqq_signals.csv',
    'TSLA': '../data/tsla_signals.csv',
}

dfs = {}
for ticker, path in signal_files.items():
    if Path(path).exists():
        df = pd.read_csv(path)
        if not df.empty:
            dfs[ticker] = df
            print(f"{ticker}: {len(df)} flagged contracts, "
                  f"ATM IV={df['atm_iv'].median():.1f}%")
    else:
        print(f"{ticker}: signals file not found")


### 1a. Put skew — IV vs moneyness (strike/spot)

In [ ]:
# For each ticker, pick the nearest liquid expiry with enough contracts
# and plot IV vs moneyness for puts

colors = {'SPY': 'steelblue', 'QQQ': 'darkorange', 'AAPL': 'seagreen', 'TSLA': 'tomato'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for kind, ax in zip(('put', 'call'), axes):
    for ticker, df in dfs.items():
        sub = df[df['kind'] == kind].copy()
        if sub.empty:
            continue

        # Pick expiry with most contracts
        best_expiry = sub.groupby('expiry_date').size().idxmax()
        sub = sub[sub['expiry_date'] == best_expiry].copy()

        # Get spot from atm_iv context — use strike where iv_vs_atm is smallest
        atm_idx = sub['iv_vs_atm'].abs().idxmin()
        spot_approx = sub.loc[atm_idx, 'strike']

        sub['moneyness'] = sub['strike'] / spot_approx
        sub = sub.sort_values('moneyness')

        ax.plot(sub['moneyness'], sub['market_iv'],
                'o-', color=colors[ticker], lw=2, ms=4,
                label=f"{ticker} ({best_expiry})")

    ax.axvline(1.0, color='grey', ls=':', alpha=0.5, label='ATM')
    ax.set_xlabel('Moneyness (Strike / Spot)')
    ax.set_ylabel('Implied Volatility (%)')
    ax.set_title(f'{kind.capitalize()} smile — nearest liquid expiry')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Cross-ticker IV smile comparison', fontsize=13)
plt.tight_layout()
plt.savefig('../notebooks/smile_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


### 1b. Skew summary — put vs call skew by ticker

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

tickers = list(dfs.keys())
x = np.arange(len(tickers))
width = 0.35

put_skews  = [dfs[t][dfs[t]['kind']=='put']['iv_vs_atm'].mean()  if t in dfs else 0 for t in tickers]
call_skews = [dfs[t][dfs[t]['kind']=='call']['iv_vs_atm'].mean() if t in dfs else 0 for t in tickers]

bars1 = ax.bar(x - width/2, put_skews,  width, label='Put skew (mean iv vs ATM)',  color='tomato',    alpha=0.8)
bars2 = ax.bar(x + width/2, call_skews, width, label='Call skew (mean iv vs ATM)', color='steelblue', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(tickers)
ax.set_ylabel('Mean IV premium vs ATM (pp)')
ax.set_title('Put vs Call skew by ticker')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(0, color='black', lw=0.8)

plt.tight_layout()
plt.savefig('../notebooks/skew_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSkew summary:")
for ticker in tickers:
    if ticker not in dfs:
        continue
    df = dfs[ticker]
    put_mean  = df[df['kind']=='put']['iv_vs_atm'].mean()
    call_mean = df[df['kind']=='call']['iv_vs_atm'].mean()
    dominant  = 'PUT skew' if put_mean > call_mean else 'CALL skew'
    print(f"  {ticker}: put={put_mean:.1f}pp, call={call_mean:.1f}pp → {dominant}")


## 2. Skew time series

In [ ]:
history_path = '../data/skew_history.csv'

if not Path(history_path).exists():
    print("No history yet — skew_history.csv will be created after the first pipeline run.")
else:
    hist = pd.read_csv(history_path, parse_dates=['date'])
    print(f"History: {len(hist)} rows, {hist['date'].nunique()} trading days")
    print(hist.tail(8).to_string(index=False))


In [ ]:
if Path(history_path).exists():
    hist = pd.read_csv(history_path, parse_dates=['date'])
    tickers_hist = hist['ticker'].unique()

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.flatten()

    for i, ticker in enumerate(tickers_hist):
        ax = axes[i]
        sub = hist[hist['ticker'] == ticker].sort_values('date')

        ax.plot(sub['date'], sub['atm_iv'],
                '-o', color='black', lw=2, ms=4, label='ATM IV')
        if sub['put_skew_mean'].notna().any():
            ax.plot(sub['date'], sub['put_skew_mean'],
                    '-s', color='tomato', lw=1.5, ms=4, label='Put skew (mean)')
        if sub['call_skew_mean'].notna().any():
            ax.plot(sub['date'], sub['call_skew_mean'],
                    '-^', color='steelblue', lw=1.5, ms=4, label='Call skew (mean)')

        ax.set_title(ticker)
        ax.set_xlabel('Date')
        ax.set_ylabel('Vol / Skew (%)')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=30)

    plt.suptitle('ATM IV and skew time series', fontsize=13)
    plt.tight_layout()
    plt.savefig('../notebooks/skew_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()


## 3. Put skew term structure — SPY

In [ ]:
# For SPY, plot how put skew varies across expiries (term structure of skew)
if 'SPY' in dfs:
    spy = dfs['SPY']
    puts = spy[spy['kind'] == 'put'].copy()

    term = (
        puts.groupby('expiry_date')
        .agg(
            expiry=('expiry', 'first'),
            put_skew_mean=('iv_vs_atm', 'mean'),
            put_skew_max=('iv_vs_atm', 'max'),
            atm_iv=('atm_iv', 'first'),
            n=('iv_vs_atm', 'count'),
        )
        .reset_index()
        .sort_values('expiry')
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(term['expiry'], term['put_skew_mean'], 'o-',
                 color='tomato', lw=2, ms=6, label='Mean put skew')
    axes[0].plot(term['expiry'], term['put_skew_max'], 's--',
                 color='tomato', lw=1.5, ms=5, alpha=0.6, label='Max put skew')
    axes[0].set_xlabel('Time to expiry (years)')
    axes[0].set_ylabel('IV premium vs ATM (pp)')
    axes[0].set_title('SPY put skew term structure')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(term['expiry'], term['atm_iv'], 'o-',
                 color='steelblue', lw=2, ms=6)
    axes[1].set_xlabel('Time to expiry (years)')
    axes[1].set_ylabel('ATM IV (%)')
    axes[1].set_title('SPY ATM IV term structure')
    axes[1].grid(alpha=0.3)

    plt.suptitle('SPY volatility term structure', fontsize=13)
    plt.tight_layout()
    plt.savefig('../notebooks/spy_term_structure.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(term[['expiry_date','expiry','atm_iv','put_skew_mean','put_skew_max','n']].to_string(index=False))
